# Unit 2 — Credit Card Fraud Detection

XGBoost + SMOTE + threshold tuning + feature importance, compared with SVM, using the Kaggle ULB credit-card dataset.

## 1. Install/import libraries

In [ ]:
# If needed in Colab:
# !pip -q install xgboost imbalanced-learn

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, precision_score, recall_score, f1_score
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

## 2. Load Kaggle dataset
Download `creditcard.csv` from Kaggle and keep it beside this notebook.

In [ ]:
df=pd.read_csv("creditcard.csv")
print("Shape:",df.shape)
display(df.head())

## 3. Check imbalance

In [ ]:
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True).mul(100).round(4))
print("Missing values:",df.isnull().sum().sum())

## 4. Prepare data

In [ ]:
X=df.drop(columns=["Class"]).copy()
y=df["Class"].astype(int)
X=X.replace([np.inf,-np.inf],np.nan).fillna(X.median(numeric_only=True)).fillna(0)

## 5. Train-test split and scaling

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,random_state=42,stratify=y
)
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)
print(X_train.shape,X_test.shape)

## 6. SMOTE — training data only

In [ ]:
print("Before SMOTE:")
print(y_train.value_counts())
smote=SMOTE(random_state=42)
X_train_sm,y_train_sm=smote.fit_resample(X_train_s,y_train)
print("\nAfter SMOTE:")
print(pd.Series(y_train_sm).value_counts())

## 7. Train XGBoost

In [ ]:
xgb=XGBClassifier(
    n_estimators=250,max_depth=6,learning_rate=0.08,
    subsample=0.8,colsample_bytree=0.8,
    objective="binary:logistic",eval_metric="logloss",
    random_state=42,n_jobs=-1
)
xgb.fit(X_train_sm,y_train_sm)
prob=xgb.predict_proba(X_test_s)[:,1]
pred=(prob>=0.50).astype(int)
print("ROC-AUC:",round(roc_auc_score(y_test,prob),4))
print("PR-AUC:",round(average_precision_score(y_test,prob),4))
print(classification_report(y_test,pred,digits=4))

## 8. Confusion matrix

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix(y_test,pred),
    display_labels=["Genuine","Fraud"]
).plot()
plt.title("XGBoost Confusion Matrix")
plt.show()

## 9. Decision-threshold tuning

In [ ]:
rows=[]
for t in np.arange(0.10,0.91,0.05):
    p=(prob>=t).astype(int)
    rows.append([t,precision_score(y_test,p,zero_division=0),
                 recall_score(y_test,p,zero_division=0),
                 f1_score(y_test,p,zero_division=0)])
threshold_df=pd.DataFrame(rows,columns=["Threshold","Precision","Recall","F1"])
display(threshold_df.round(4))
best_t=float(threshold_df.loc[threshold_df.F1.idxmax(),"Threshold"])
print("Selected threshold:",best_t)

## 10. Evaluate tuned threshold

In [ ]:
pred_tuned=(prob>=best_t).astype(int)
print(classification_report(y_test,pred_tuned,digits=4))

## 11. ROC and Precision-Recall curves

In [ ]:
fpr,tpr,_=roc_curve(y_test,prob)
p,r,_=precision_recall_curve(y_test,prob)

plt.figure(figsize=(7,5))
plt.plot(fpr,tpr,label=f"ROC-AUC={roc_auc_score(y_test,prob):.4f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend(); plt.grid(); plt.show()

plt.figure(figsize=(7,5))
plt.plot(r,p,label=f"PR-AUC={average_precision_score(y_test,prob):.4f}")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(); plt.show()

## 12. Feature importance

In [ ]:
importance=pd.Series(xgb.feature_importances_,index=X.columns).sort_values(ascending=False)
display(importance.head(15).to_frame("Importance"))
importance.head(15).sort_values().plot(kind="barh",figsize=(8,6))
plt.title("Top 15 XGBoost Feature Importances")
plt.xlabel("Importance"); plt.show()

## 13. Baseline SVM

In [ ]:
svm=SVC(kernel="rbf",probability=True,class_weight="balanced",random_state=42)
svm.fit(X_train_s,y_train)
svm_prob=svm.predict_proba(X_test_s)[:,1]
svm_pred=(svm_prob>=0.50).astype(int)

print("SVM ROC-AUC:",round(roc_auc_score(y_test,svm_prob),4))
print("SVM PR-AUC:",round(average_precision_score(y_test,svm_prob),4))
print(classification_report(y_test,svm_pred,digits=4))

## 14. Compare XGBoost vs SVM

In [ ]:
comparison=pd.DataFrame({
    "Model":["XGBoost","SVM"],
    "ROC-AUC":[roc_auc_score(y_test,prob),roc_auc_score(y_test,svm_prob)],
    "PR-AUC":[average_precision_score(y_test,prob),average_precision_score(y_test,svm_prob)],
    "Precision":[precision_score(y_test,pred_tuned,zero_division=0),precision_score(y_test,svm_pred,zero_division=0)],
    "Recall":[recall_score(y_test,pred_tuned,zero_division=0),recall_score(y_test,svm_pred,zero_division=0)],
    "F1":[f1_score(y_test,pred_tuned,zero_division=0),f1_score(y_test,svm_pred,zero_division=0)]
})
display(comparison.round(4))

## 15. Final interpretation

In [ ]:
print("XGBoost uses SMOTE to handle training-set class imbalance.")
print("Threshold tuning changes the precision-recall trade-off.")
print("PR-AUC is particularly informative for highly imbalanced fraud detection.")
print("Feature importance helps interpret which variables contribute most to the model.")

**Kaggle source:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud